In [1]:
"""
Ovarian CT Classification - Malignancy & Subtype Prediction
===========================================================
Architecture: ResNet18 / ResNet50 / ViT / Hybrid (Patient-level Attention Classifier)
Strategy: 5-fold Cross Validation | Cascade (Malignancy -> Subtype)
Slice-level auxiliary loss
"""

'\nOvarian CT Classification - Malignancy & Subtype Prediction\n===========================================================\nArchitecture: ResNet18 / ResNet50 / ViT / Hybrid (Patient-level Attention Classifier)\nStrategy: 5-fold Cross Validation | Cascade (Malignancy -> Subtype)\nSlice-level auxiliary loss\n'

In [2]:
import os
import random
import numpy as np
from collections import OrderedDict, defaultdict, Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.models import ViT_B_16_Weights
import torchvision.models as tv_models
from pathlib import Path
from sklearn.metrics import (accuracy_score, roc_auc_score, 
                             f1_score, recall_score, confusion_matrix)
from scipy.stats import mode as scipy_mode

import warnings
warnings.filterwarnings("ignore")

In [6]:
# 0. Config
DATA_ROOT = Path("/tf/nasw/dataset001/preprocessed/npz_us")
OUTPUT_ROOT = Path("./Cascade_models")

PRETRAINED = {
    "resnet18": Path("/tf/pretrained_model/resnet18-f37072fd.pth"),
    "resnet50": Path("/tf/pretrained_model/resnet50-11ad3fa6.pth"),
    "vit": Path("/tf/pretrained_model/vit_b_16-c867db91.pth"),
}

RESNET_ARCH = "resnet50" # "resnet50 or "resnet18"

FOLDS = [1, 2, 3, 4, 5]
NUM_FOLDS = 5
BATCH_SIZE = 8
NUM_WORKERS = 4
LEARNING_RATE = 1e-4
LR_BACKBONE = 1e-4
LR_HEAD = 1e-4
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MALIGNANT_MAP: dict[int, int] = {}
NUM_MALIGNANT_SUBTYPES: int = 0

def infer_malignant_type_ids(data_root, folds):
    label_set_by_type = defaultdict(set)

    for fold_idx in FOLDS:
        for split_name in ["train", "val"]:
            npz_path = Path(data_root) / f"fold_{fold_idx}_{split_name}.npz"
            data = np.load(npz_path, allow_pickle=False)

            labels = data["labels"].astype(int)
            tumor_types = data["tumor_types"].astype(int)

            for y, t in zip(labels, tumor_types):
                label_set_by_type[int(t)].add(int(y))
                
        conflicts = {
            t: sorted(list(v))
            for t, v in label_set_by_type.items()
            if len(v) > 1
        }

        if len(conflicts) > 0:
            raise ValueError(
                f"tumor_type가 양성/악성 양쪽에 섞여있음: {conflicts}\n"
            )
        malignant_type_ids = sorted([
            t for t, labs in label_set_by_type.items()
            if labs == {1}
        ])

        print("malignant_type_ids:", malignant_type_ids)
        return malignant_type_ids


MALIGNANT_TYPE_IDS = infer_malignant_type_ids(DATA_ROOT, FOLDS)
MALIGNANT_MAP = {tid: i for i, tid in enumerate(MALIGNANT_TYPE_IDS)}
NUM_MALIGNANT_SUBTYPES = len(MALIGNANT_TYPE_IDS)
print("NUM_MALIGNANT_SUBTYPES:", NUM_MALIGNANT_SUBTYPES)


HIDDEN_DIM = 512
DROPOUT = 0.3
SUBTYPE_LOSS_WEIGHT= 0.2 # patient-level subtype loss weight
SLICE_SUBTYPE_LOSS_WEIGHT = 0.03 # slice-level auxiliary subtype loss weight
FOCAL_GAMMA = 1.0 # 높을수록 easy example 더 강하게 억제
LABEL_SMOOTHING = 0.1

WEIGHT_DECAY = 1e-4
PATIENCE = 7
MIN_DELTA = 0.001
THRESHOLD = 0.5


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

malignant_type_ids: [0, 1, 3, 5, 7]
NUM_MALIGNANT_SUBTYPES: 5


In [7]:
# 1. Dataset 정의

class OvarianCTNPZDataset(Dataset):
    """
    Loads pre-processed CT slices from a .npz file,
    Expected keys: 'images', 'labels', 'tumor_types', 'patients'
    """
    
    def __init__(self, npz_path: Path):
        super().__init__()
        self.npz = np.load(npz_path, allow_pickle=False)

        self.images = self.npz["images"]
        self.labels = self.npz["labels"].astype(np.float32)
        self.tumor_types = self.npz["tumor_types"].astype(np.int64)
        self.patient_ids = (self.npz["patient_ids"] if "patient_ids" in self.npz.files 
                            else np.arange(len(self.labels)))


        # ImageNet normalization constants
        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.images[idx]
        y = self.labels[idx]
        tumor_type = int(self.tumor_types[idx])


        # (S, H, W, C) -> (S, C, H, W), normalize to [0,1] then ImageNet-nomalize
        x = torch.from_numpy(x).float().permute(0, 3, 1, 2).contiguous()
        x = x / 255.0
        x = (x-self.mean) / self.std
        

        if y == 1:
            if tumor_type not in MALIGNANT_MAP:
                raise ValueError(
                    f"malignant sample인데 tumor_type={tumor_type}가 MAP에 없음"
                )
            mal_subtype = MALIGNANT_MAP[tumor_type]
            mal_mask = True
        else:
            mal_subtype = -1
            mal_mask = False
            
        return {
            "image": x,
            "label": torch.tensor(y, dtype = torch.float32),
            "mal_subtype": torch.tensor(mal_subtype, dtype=torch.long),
            "mal_mask": torch.tensor(mal_mask, dtype=torch.bool),
        }

In [8]:
# 2. Encoders
def _load_local_weights(model: nn.Module, path: Path, strict: bool = True) -> nn.Module:
    state = torch.load(path, map_location="cpu")
    model.load_state_dict(state, strict=strict)
    return model

def build_resnet_encoder(arch: str = RESNET_ARCH,
                           freeze_backbone: bool = True, 
                           unfreeze_layer4: bool = True) -> tuple[nn.Module, int]:
    if arch == "resnet18":
        encoder = models.resnet18(weights=None)
    else:
        encoder = models.resnet50(weights=None)

    weight_path = PRETRAINED[arch]
    if weight_path.exists():
        encoder = _load_local_weights(encoder, weight_path)
    else:
        print(f"  [WARN] pretrained weights 없음: {weight_pah} - random init 사용")
        
    feat_dim = encoder.fc.in_features
    encoder.fc = nn.Identity()

    if freeze_backbone:
        for p in encoder.parameters():
            p.requires_grad = False
        if unfreeze_layer4:
            for p in encoder.layer4.parameters():
                p.requires_grad = True
    
    return encoder, feat_dim

def build_resnet50_encoder(freeze_backbone: bool = True,
                           unfreeze_layer4: bool = True) -> tuple[nn.Module, int]:
    return build_resnet_encoder("resnet50", freeze_backbone, unfreeze_layer4)

def build_vit_encoder(freeze_backbone: bool = True) -> tuple[nn.Module, int]:
    
    encoder = tv_models.vit_b_16(weights=None)
    encoder = _load_local_weights(encoder, PRETRAINED["vit"])
    
    feat_dim = encoder.heads.head.in_features
    encoder.heads.head = nn.Identity()
    
    if freeze_backbone:
        for p in encoder.parameters():
            p.requires_grad = False
            
    return encoder, feat_dim

In [9]:
# 3. Classifers
# slice encoder + patient-level pooling wrapper

class PatientSliceAttentionClassifier(nn.Module):
    """
    Aggregates per-slice features via attention pooling, then:
    - shared branch -> malignancy head (binary)
    - private brach -> subtype head (multi-class, malignant only)

    Forward input: x (B, S, C, H, W)
    Forward output: dict_with 'malignancy_logits', 'subtype_logits'
    """
    
    def __init__(self, encoder: nn.Module, feat_dim: int, 
                 hidden_dim: int = HIDDEN_DIM, 
                 num_subtypes: int = NUM_MALIGNANT_SUBTYPES,
                 dropout: float = DROPOUT):
        super().__init__()
        self.encoder = encoder
        
        # Attention pooling: feat_dim -> feat_dim//2 -> 1
        self.attn = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2),
            nn.Tanh(),
            nn.Linear(feat_dim // 2, 1),
        )

        # Shared representation -> malignancy head
        self.shared = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.malignancy_head = nn.Linear(hidden_dim, 1)

        # Patient-level subtype branch
        self.subtype_private = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
        )
        self.subtype_head = nn.Linear(hidden_dim, num_subtypes)

        # Slice-level auxiliary subtype branch
        # 슬라이스 각각을 독립 샘플로 학습 -> 소수 클래스 gradient 8배 증가
        self.slice_subtype_private = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
        )
        self.slice_subtype_head = nn.Linear(hidden_dim, num_subtypes)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        # x: (B, S, C, H, W)
        B, S, C, H, W = x.shape

        # Encode all slices in parallel
        feat = self.encoder(x.view(B * S, C, H, W)).view(B, S, -1) # (B, S, feat_dim)
        
        # Attention-weighted patient-level pooling (malignancy 용)
        attn_weight = torch.softmax(self.attn(feat), dim=1) # (B, S, 1)
        pooled = (feat * attn_weight).sum(dim=1) # (B, feate_dim)
        
        # Malignancy: patient-level (attention pooling 유지)
        z = self.shared(pooled)
        malignancy_logits = self.malignancy_head(z).squeeze(1) # (B,)

        # Subtype: slice-level (각 슬라이스를 독립 샘플로)
        # (B, S, feat_dim) -> (B*S, feat_dim) -> (B, S, num_subtypes)
        z_slice = self.slice_subtype_private(feat.view(B * S, -1))
        slice_sub_logits = self.slice_subtype_head(z_slice).view(B, S, -1) # (B, S, C)

        # Patient-level subtype: attention 가중 평균 (보조 loss용, 퓨전용 벡터)
        # attn_weight: (B, S, 1) -> subtype logits 가중 평균
        subtype_logits = (slice_sub_logits * attn_weight).sum(dim=1) # (B, num_subtyes)
        
        return {
            "malignancy_logits": malignancy_logits,
            "subtype_logits": subtype_logits, # attention 가중 평균 (patient-level)
            "slice_sub_logits": slice_sub_logits, # 슬라이스별 logits (학습 + majority voting)
        }

class CNNTransformerHybridPatient(nn.Module):
    """ 
    Hybrid model: ResNet50 (per-slice CNN encoder) + Transformer (inter-slice context).
    A learnable CLS token aggregates slice-level features for patient-level prediction.

    Architecture:
        ResNet50 -> proj -> [CLS, slice_1, ..., slice_S] + pos_embed
                 -> TransformerEncoder -> CLS output
                 -> shared -> malignancy_head / subtype_head
    """
    
    def __init__(self, d_model: int = 512, num_layers: int = 2, nhead: int = 8,
                 dim_feedforward: int = 1024, dropout: float = 0.1, 
                 load_pretrained: bool = True, freeze_backbone: bool = True, 
                 unfreeze_layer4: bool = True):
        super().__init__()

        # CNN backbone (ResNet50)
        backbone = models.resnet50(weights=None)
        if load_pretrained:
            backbone = _load_local_weights(backbone, PRETRAINED["resnet50"])
        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.encoder = backbone

        if freeze_backbone:
            for p in self.encoder.parameters():
                p.requires_grad = False
            if unfreeze_layer4:
                for p in self.encoder.layer4.parameters():
                    p.requires_grad = True

        # Project CNN features into Transformer d_model space
        self.proj = nn.Linear(feat_dim, d_model)

        # Learnalbe CLS token and positional embedding
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + 8, d_model))

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers= num_layers)
        self.norm = nn.LayerNorm(d_model)

        # Shared head + task-specific heads
        self.shared = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.malignancy_head = nn.Linear(512, 1)
        self.subtype_head = nn.Linear(512, NUM_MALIGNANT_SUBTYPES)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        B, S, C, H, W = x.shape
        
        # Per-slice CNN encoding
        feat = self.encoder(x.view(B * S, C, H, W)).view(B, S, -1) # (B, S, feat_dim)
        feat = self.proj(feat) # (B, S, d_model)

        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1) # (B, 1, d_model)
        tokens = torch.cat([cls, feat], dim=1) # (B, 1+S, d_model)

        # Add positional embedding (trim to actual sequence length)
        tokens = tokens + self.pos_embed[:, :tokens.size(1), :]

        # Transformer + LayerNorm
        tokens = self.transformer(tokens)
        tokens = self.norm(tokens)

        # CLS token output as patient representation
        cls_out = tokens[:, 0, :] # (B, d_model)
        z = self.shared(cls_out)

        
        malignancy_logits = self.malignancy_head(z).squeeze(1)
        subtype_logits = self.subtype_head(z)
        
        return {
            "malignancy_logits": malignancy_logits, # (B,)
            "subtype_logits": subtype_logits # (B, num_subtypes)
        }

In [10]:
# 4. Factory

def build_model(arch: str = "resnet50") -> nn.Module:
    """
    arch options:
        - "resnet50" : ResNet50 + Attention Polling Classifier
        - "resnet18" : ResNet18 + Attention Pooling Classifier (경량 모델)
        - "vit" : Vit-B/16 + Attention Pooling Classifier
        - "hybrid" : ResNet50 + Transformer (CLS token) Classifer
    """

    if arch == "hybrid":
        return CNNTransformerHybridPatient().to(DEVICE)

    if arch in ("resnet50", "resnet18"):
        encoder, feat_dim = build_resnet_encoder(arch)
    elif arch == "vit":
        encoder, feat_dim = build_vit_encoder()
    else:
        raise ValueError( f"Unknown arch: {arch}")
    
    model = PatientSliceAttentionClassifier(encoder=encoder, feat_dim=feat_dim)
    return model.to(DEVICE)

In [11]:
# 5. Loss

class FocalLoss(nn.Module):
    """
    Multi-class Focal Loss.
        FL(p_t) = -alpha_t * (1-p_t)^gamma * log(p_t)
    - alpha (class_weight): 다수 클래스 억제 (ENS weight 그대로 사용)
    - gamma: easy example 억제 (기본값 2.0)
    - smoothing: 정답 타겟을 1.0 -> (1-smoothing)으로 낮춰 과적합 억제
    """
    def __init__(self, weight: torch.Tensor, gamma: float = 2.0,
                 smoothing: float = 0.0):
        super().__init__()
        self.register_buffer("weight", weight.float())
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        num_classes = logits.size(1)
        log_prob = nn.functional.log_softmax(logits, dim=1) # (N, C)
        prob = log_prob.exp() # (N, C)

        # alpha: 각 샘플의 클래스 weight
        alpha = self.weight[targets] # (N,)

        # p_t: 정답 클래스의 확률
        p_t = prob[torch.arange(len(targets)), targets] # (N,)

        # Focal weight
        focal_w = (1.0 - p_t) ** self.gamma # (N,)

        if self.smoothing > 0.0:
            # soft target: 정답=1-s, 나머지=s/(C-1)
            smooth_val = self.smoothing / (num_classes - 1)
            soft_target = torch.full_like(log_prob, smooth_val)
            soft_target.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
            loss_per_sample = -(soft_target * log_prob).sum(dim=1)
        else:
            loss_per_sample = -log_prob[torch.arange(len(targets)), targets]

        loss = (alpha * focal_w * loss_per_sample).mean()
        return loss


class CascadeCriterion(nn.Module):
    """
    Cascade loss:
        total = loss_malignancy + subtype_loss_weight * loss_subtype
    Subtype loss: Focal Loss (gamma=2) + ENS class weights
        -> 다수 클래스 (easy example) 억제 + 소수 클래스 강조
    """
    
    def __init__(self, pos_weight: torch.Tensor, 
                 subtype_class_weights: torch.Tensor,
                 subtype_loss_weight: float = SUBTYPE_LOSS_WEIGHT,
                 focal_gamma: float = 2.0,
                 label_smoothing: float = 0.0,
                 slice_subtype_loss_weight: float = SLICE_SUBTYPE_LOSS_WEIGHT):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.focal = FocalLoss(weight=subtype_class_weights, gamma=focal_gamma, smoothing=label_smoothing)
        self.subtype_loss_weight = subtype_loss_weight
        self.slice_subtype_loss_weight = slice_subtype_loss_weight

    def forward(self, outputs: dict, batch: dict) -> dict[str, torch.Tensor]:
        labels = batch["label"].to(DEVICE).view(-1)
        mal_mask = batch["mal_mask"].to(DEVICE).view(-1)
        mal_subtype = batch["mal_subtype"].to(DEVICE).view(-1)

        loss_m = self.bce(outputs["malignancy_logits"], labels)

        # Patient-level subtype loss
        if mal_mask.any():
            loss_s = self.focal(
                outputs["subtype_logits"][mal_mask],
                mal_subtype[mal_mask]
            )

        else:
            loss_s = torch.tensor(0.0, device = DEVICE)

        # Slice-level auxiliary subtype loss
        # slice_sub_logits: (B, S, num_subtypes) -> 악성 환자의 모든 슬라이스
        # 각 슬라이스에 동일한 patient-level subtype label 부여
        if mal_mask.any() and "slice_sub_logits" in outputs:
            B, S, C_sub = outputs["slice_sub_logits"].shape
            # 악성 환자만 추출: (N_mal, S, num_subtypes)
            slice_logits_mal = outputs["slice_sub_logits"][mal_mask]
            # (N_mal, S, num_subtypes) -> (N_mal*S, num_subtypes)
            slice_logits_flat = slice_logits_mal.view(-1, C_sub)
            # label을 S번 반복: (N_mal,) -> (N_mal*S,)
            slice_labels_flat = mal_subtype[mal_mask].unsqueeze(1).expand(-1, S).reshape(-1)

            loss_slice = self.focal(slice_logits_flat, slice_labels_flat)
        else:
            loss_slice = torch.tensor(0.0, device=DEVICE)
        
    
        losses_with_grad = []
        if loss_m.requires_grad:
            losses_with_grad.append(loss_m)
        if loss_s.requires_grad and self.subtype_loss_weight > 0:
            losses_with_grad.append(self.subtype_loss_weight * loss_s)
        if loss_slice.requires_grad and self.slice_subtype_loss_weight > 0:
            losses_with_grad.append(self.slice_subtype_loss_weight * loss_slice)
            
        loss_total = sum(losses_with_grad) if losses_with_grad else (loss_m + self.subtype_loss_weight * loss_s).detach()
        
        return {
            "loss_total": loss_total,
            "loss_malignancy": loss_m.detach(),
            "loss_subtype": loss_s.detach() if loss_s.requires_grad else loss_s,
            "loss_slice": loss_slice.detach() if loss_slice.requires_grad else loss_slice,
        }

In [12]:
# 6. DataLoader factory (per fold)

def _compute_class_weights(labels: np.ndarray,
                           tumor_types: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute pos_weight (malignancy) and subtype class weight (ENS).
    """
    neg = (labels == 0).sum()
    pos = (labels == 1).sum()
    pos_weight = torch.tensor([neg / max(pos, 1)], device = DEVICE, dtype = torch.float32)

    # actual_mal_types = [int(t) for y, t in zip(labels.astype(int), tumor_types.astype(int)) if y == 1]
    # actual_counts = Counter(actual_mal_types)
    # unmapped = [t for t in actual_counts if t not in MALIGNANT_MAP]
    # print(f" MALIGNANT_MAP: {MALIGNANT_MAP}")
    # print(f" 실제 악성 tumor_type 분포: {dict(sorted(actual_counts.items()))}")

    # if unmapped:
    #     print(f" [경고] MALIGNANT_MAP에 없는 tumor_type: {unmapped}")

    # Effective Numver of Samples reweighting
    subtype_counts = np.zeros(NUM_MALIGNANT_SUBTYPES, dtype=np.float64)
    for y, t in zip(labels.astype(int), tumor_types.astype(int)):
        if y == 1 and int(t) in MALIGNANT_MAP:
            subtype_counts[MALIGNANT_MAP[t]] += 1.0
    
    subtype_counts = np.maximum(subtype_counts, 1.0)
    beta = 0.9999
    effective_num = (1.0 - np.power(beta, subtype_counts)) / (1.0 - beta)
    weights_ens = 1.0 / effective_num
    weights_ens = weights_ens / weights_ens.min()

    print(f" subtype_counts = {subtype_counts.tolist()}")
    print(f" effective_num = {effective_num.tolist()}")
    print(f" weights_ens (raw) = {weights_ens.tolist()}")
    print(f" subtype_class_weights (normalized) = {weights_ens.round(4).tolist()}")
    
    subtype_class_weights = torch.tensor(weights_ens, dtype=torch.float32, device=DEVICE)

    return pos_weight, subtype_class_weights


def make_fold_loaders(fold_idx: int) -> tuple:
    train_npz = DATA_ROOT / f"fold_{fold_idx}_train.npz"
    val_npz = DATA_ROOT / f"fold_{fold_idx}_val.npz"
    assert train_npz.exists(), f"Missing: {train_npz}"
    assert val_npz.exists(), f"Missing: {val_npz}"

    train_dataset = OvarianCTNPZDataset(train_npz)
    val_dataset = OvarianCTNPZDataset(val_npz)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle = True, num_workers = NUM_WORKERS, pin_memory = True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                            shuffle = False,num_workers = NUM_WORKERS, pin_memory = True)

    pos_weight, subtype_class_weights = _compute_class_weights(
        train_dataset.labels.astype(int),
        train_dataset.tumor_types.astype(int),
    )

    print(f"[FOLD {fold_idx}] "
          f"benign(0)={(train_dataset.labels==0).sum()} "
          f"malignant(1)={(train_dataset.labels==1).sum()} "
          f"pos_weight={pos_weight.item():.4f}")
    print(f"  subtype_class_weights = {subtype_class_weights.detach().cpu().numpy().round(4).tolist()}")

    return train_loader, val_loader, pos_weight, subtype_class_weights, SUBTYPE_LOSS_WEIGHT

In [13]:
# 7. Train / Eval loops

def _compute_binary_metrics(labels: list, probs: list,
                            threshold: float = THRESHOLD) -> dict:
    """
    ACC, AUC, F1, Recall, Confusion Matrix for malignancy.
    """
    preds = (np.array(probs) >= threshold).astype(int)
    labs = np.array(labels).astype(int)
    auc = roc_auc_score(labs, probs) if len(set(labs)) > 1 else float("nan")

    return {
        "acc": float(accuracy_score(labs, preds)),
        "auc": float(auc),
        "f1": float(f1_score(labs, preds, zero_division=0)),
        "recall": float(recall_score(labs, preds, zero_division=0)),
        "cm": confusion_matrix(labs, preds).tolist(),
    }

def _compute_multiclass_metrics(true: np.ndarray, preds: np.ndarray) -> dict:
    """
    ACC, marco-F1, Confusion Matrix for subtype.
    """
    return {
        "acc": float(accuracy_score(true, preds)),
        "macro_f1": float(f1_score(true, preds, average="macro", zero_division=0)),
        "cm": confusion_matrix(true, preds).tolist(),
    }
    
def train_one_epoch(model: nn.Module, loader: DataLoader, 
                    criterion: CascadeCriterion, 
                    optimizer: torch.optim.Optimizer, 
                    scaler: torch.cuda.amp.GradScaler) -> dict:
    model.train()
    running_total = running_m = running_s = running_slice = 0.0
    all_label, all_mal_prob, all_mal_pred = [], [], []
    all_mal_mask, all_subtype_true, all_subtype_pred = [], [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            outputs = model(imgs)
            loss_dict = criterion(outputs, batch)
            loss = loss_dict["loss_total"]

        if loss.requires_grad:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = imgs.size(0)
        running_total += loss_dict["loss_total"].item() * bs
        running_m += loss_dict["loss_malignancy"].item() * bs
        running_s += loss_dict["loss_subtype"].item() * bs
        running_slice += loss_dict["loss_slice"].item() * bs

        label_np = batch["label"].detach().cpu().numpy().astype(int).ravel()
        mal_mask_np = batch["mal_mask"].detach().cpu().numpy().astype(bool).ravel()
        mal_prob_np = torch.sigmoid(outputs["malignancy_logits"]).detach().cpu().numpy().ravel()
        mal_pred_np = (mal_prob_np >= THRESHOLD).astype(int)

        subtype_true_np = batch["mal_subtype"].detach().cpu().numpy().astype(int).ravel()

        # Majority Voting: slice-level 예측을 환자 단위로 집계
        if "slice_sub_logits" in outputs:
            slice_preds = outputs["slice_sub_logits"].argmax(dim=2).detach().cpu().numpy()
            subtype_pred_np = np.array([
                scipy_mode(slice_preds[i], keepdims=True).mode[0]
                for i in range(slice_preds.shape[0])
            ]).astype(int).ravel()
        else:
            subtype_pred_np = outputs["subtype_logits"].argmax(dim=1).detach().cpu().numpy().astype(int).ravel()

        all_label.extend(label_np.tolist())
        all_mal_prob.extend(mal_prob_np.tolist())
        all_mal_pred.extend(mal_pred_np.tolist())
        all_mal_mask.extend(mal_mask_np.tolist())
        all_subtype_true.extend(subtype_true_np.tolist())
        all_subtype_pred.extend(subtype_pred_np.tolist())

    mal_metrics = _compute_binary_metrics(all_label, all_mal_prob)

    all_label = np.asarray(all_label).astype(int)
    all_mal_pred = np.asarray(all_mal_pred).astype(int)
    all_mal_mask = np.asarray(all_mal_mask).astype(bool)
    all_subtype_true = np.asarray(all_subtype_true).astype(int)
    all_subtype_pred = np.asarray(all_subtype_pred).astype(int)

    # Subtype metrics - GT malignant patient only
    oracle_true = all_subtype_true[all_mal_mask]
    oracle_pred = all_subtype_pred[all_mal_mask]
    subtype_oracle = _compute_multiclass_metrics(oracle_true, oracle_pred)

    # Cascade subtype: malignancy를 맞춘 환자만 subtype 평가
    #GT 악성 샘플 중에서, malignancy도 맞추고 subtype도 맞춰야 정답
    gt_mal_mask = (all_label == 1)
    if gt_mal_mask.sum() > 0:
        gate_recall = (all_mal_pred[gt_mal_mask] == 1).mean()
        cascade_pred = np.where(all_mal_pred[gt_mal_mask] == 1,
                                oracle_pred, -1) # 관문 실패 -> -1
        cascade_f1 = float(f1_score(oracle_true, cascade_pred,
                                    average="macro", zero_division=0))
        cascade_correct = (cascade_pred == oracle_true).astype(np.float32)
        cascade_acc = cascade_correct.mean()
    else:
        gate_recall = cascade_acc = cascade_f1 = np.nan

    n = len(loader.dataset)
    print(f" [Loss 학습]"
          f"loss_m={running_m/n:.4f}, "
          f"loss_s={running_s/n:.4f}, "
          f"loss_total={running_total/n:.4f}")

    metrics = {
        "loss": running_total / n,
        "loss_malignancy": running_m / n,
        "loss_subtype": running_s / n,
        "loss_slice": running_slice / n,
        
        "acc": mal_metrics["acc"],
        "auc": mal_metrics["auc"],
        "f1":mal_metrics["f1"],
        "recall": mal_metrics["recall"],
        "cm": mal_metrics["cm"],

        "subtype_oracle_acc": subtype_oracle["acc"],
        "subtype_oracle_macro_f1": subtype_oracle["macro_f1"],
        "subtype_oracle_cm": subtype_oracle["cm"],

        "subtype_cascade_acc": float(cascade_acc),
        "subtype_cascade_f1": float(cascade_f1),
        "subtype_gate_recall": float(gate_recall),
    }

    return metrics


@torch.no_grad()
def eval_one_epoch(model: nn.Module, loader: DataLoader, 
                   criterion: CascadeCriterion) -> dict:
    model.eval()
    running_total = running_m = running_s = running_slice = 0.0
    all_label, all_mal_prob, all_mal_pred = [], [], []
    all_mal_mask, all_subtype_true, all_subtype_pred = [], [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            outputs = model(imgs)
            loss_dict = criterion(outputs, batch)

        bs = imgs.size(0)
        running_total += loss_dict["loss_total"].item() * bs
        running_m += loss_dict["loss_malignancy"].item() * bs
        running_s += loss_dict["loss_subtype"].item() * bs
        running_slice += loss_dict["loss_slice"].item() * bs

        label_np = batch["label"].detach().cpu().numpy().astype(int).ravel()
        mal_mask_np = batch["mal_mask"].detach().cpu().numpy().astype(bool).ravel()

        mal_prob_np = torch.sigmoid(outputs["malignancy_logits"]).detach().cpu().numpy().ravel()
        mal_pred_np = (mal_prob_np >= THRESHOLD).astype(int)

        subtype_true_np = batch["mal_subtype"].detach().cpu().numpy().astype(int).ravel()

        # Majority Voting: slice_sub_logits가 있으면 슬라이스별 예측을 집계
        # 없으면 patient-level subtype_logits 사용 (fallback)
        if "slice_sub_logits" in outputs:
            # (B, S, num_subtypes) -> 슬라이스별 argmax -> (B, S)
            slice_preds = outputs["slice_sub_logits"].argmax(dim=2).detach().cpu().numpy() # (B, S)
            # 환자별 majority vote: 가장 많이 예측된 subtype
            subtype_pred_np = np.array([
                scipy_mode(slice_preds[i], keepdims=True).mode[0]
                for i in range(slice_preds.shape[0])
            ]).astype(int).ravel()
        else:
            subtype_pred_np = outputs["subtype_logits"].argmax(dim=1).detach().cpu().numpy().astype(int).ravel()
            
        all_label.extend(label_np.tolist())
        all_mal_prob.extend(mal_prob_np.tolist())
        all_mal_pred.extend(mal_pred_np.tolist())
        all_mal_mask.extend(mal_mask_np.tolist())
        all_subtype_true.extend(subtype_true_np.tolist())
        all_subtype_pred.extend(subtype_pred_np.tolist())

    mal_metrics = _compute_binary_metrics(all_label, all_mal_prob)

    all_label = np.asarray(all_label).astype(int)
    all_mal_pred = np.asarray(all_mal_pred).astype(int)
    all_mal_mask = np.asarray(all_mal_mask).astype(bool)
    all_subtype_true = np.asarray(all_subtype_true).astype(int)
    all_subtype_pred = np.asarray(all_subtype_pred).astype(int)

    oracle_true = all_subtype_true[all_mal_mask]
    oracle_pred = all_subtype_pred[all_mal_mask]
    subtype_oracle = _compute_multiclass_metrics(oracle_true, oracle_pred)

    gt_mal_mask = (all_label == 1)
    if gt_mal_mask.sum() > 0:
        gate_recall = (all_mal_pred[gt_mal_mask] == 1).mean()
        cascade_pred = np.where(all_mal_pred[gt_mal_mask] == 1,
                                oracle_pred, -1) # 관문 실패 -> -1
        cascade_f1 = float(f1_score(oracle_true, cascade_pred,
                                    average="macro", zero_division=0))
        cascade_correct = (cascade_pred == oracle_true).astype(np.float32)
        cascade_acc = cascade_correct.mean()
    else:
        gate_recall = cascade_acc = cascade_f1 = np.nan

    n = len(loader.dataset)

    metrics = {
        "loss": running_total / n,
        "loss_malignancy": running_m / n,
        "loss_subtype": running_s / n,
        "loss_slice": running_slice / n,
        
        "acc": mal_metrics["acc"],
        "auc": mal_metrics["auc"],
        "f1": mal_metrics["f1"],
        "recall": mal_metrics["recall"],
        "cm": mal_metrics["cm"],

        "subtype_oracle_acc": subtype_oracle["acc"],
        "subtype_oracle_macro_f1": subtype_oracle["macro_f1"],
        "subtype_oracle_cm": subtype_oracle["cm"],

        "subtype_cascade_acc": float(cascade_acc),
        "subtype_cascade_f1": float(cascade_f1),
        "subtype_gate_recall": float(gate_recall),
    }
    return metrics

In [14]:
# 8. Full CV experiment

def run_cv_experiment(arch: str = "resnet50"):
    print(f"\n{'='*60}")
    print(f"  Architecture: {arch.upper()}  | Device: {DEVICE}")
    print(f"{'='*60}")
    
    fold_aucs = []

    for fold_idx in range(1, NUM_FOLDS + 1):
        print(f"\n--- Fold {fold_idx}/{NUM_FOLDS} -----------------------------")

        train_loader, val_loader, pos_weight, subtype_class_weights, subtype_loss_weight = make_fold_loaders(fold_idx)

        model = build_model(arch)
        criterion = CascadeCriterion(pos_weight, subtype_class_weights, subtype_loss_weight, 
                                     FOCAL_GAMMA, LABEL_SMOOTHING,
                                     SLICE_SUBTYPE_LOSS_WEIGHT)
        optimizer = torch.optim.AdamW(
        [{"params": [p for n, p in model.named_parameters()
                       if p.requires_grad and "encoder" in n], "lr": LR_BACKBONE},
         {"params": [p for n, p in model.named_parameters()
                   if p.requires_grad and "encoder" not in n], "lr": LR_HEAD}],
        weight_decay=WEIGHT_DECAY,
    )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


        best_auc, best_state = 0.0, None
        best_val_loss = float("inf")
        patience_counter = 0
        
        for epoch in range(1, EPOCHS + 1):
            train_m = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
            val_m = eval_one_epoch(model, val_loader, criterion)
            scheduler.step()

            print(f"  Epoch {epoch:02d}/{EPOCHS}  "
                  f"loss={train_m['loss']:.4f}|{val_m['loss']:.4f}  "
                  f"auc={train_m['auc']:.4f}|{val_m['auc']:.4f}  "
                  f"f1={train_m['f1']:.4f}|{val_m['f1']:.4f}  "
                  f"oracle_acc={train_m['subtype_oracle_acc']:.4f}|{val_m['subtype_oracle_acc']:.4f} "
                  f"oracle_f1={train_m['subtype_oracle_macro_f1']:.4f}|{val_m['subtype_oracle_macro_f1']:.4f} "
                  f"cascade_acc={train_m['subtype_cascade_acc']:.4f}|{val_m['subtype_cascade_acc']:.4f} "
                  f"loss_slice={train_m['loss_slice']:.4f}|{val_m['loss_slice']:.4f} "
                  f"patience={patience_counter}/{PATIENCE}")

            # best model 저장 기준: val AUC
            if val_m["auc"] > best_auc:
                best_auc = val_m["auc"]
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

            # early stopping 기준: val loss (AUC보다 overfitting을 빨리 감지)
            if val_m["loss"] < best_val_loss - MIN_DELTA:
                best_val_loss = val_m["loss"]
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"  Early stopping at epoch {epoch} (patience={PATIENCE})")
                    break


        fold_aucs.append(best_auc)
        torch.save(best_state, OUTPUT_ROOT / f"{arch}_fold{fold_idx}_best.pth")
        print(f" Best val AUC: {best_auc:.4f}")

    print(f"\n{'='*60}")
    print(f"  {arch.upper()} - CV AUC: {np.mean(fold_aucs):.4f} +- {np.std(fold_aucs):.4f}")
    print(f"{'='*60}")

    return fold_aucs

In [15]:
# 9. Entry point

if __name__ == "__main__":
    results = {}
    for arch in ["resnet18", "resnet50", "vit", "hybrid"]:
        results[arch] = run_cv_experiment(arch)

    print("\n=== Summary ===")
    for arch, aucs in results.items():
        print(f" {arch:10s}: {np.mean(aucs):.4f} +- {np.std(aucs):.4f}")


  Architecture: RESNET18  | Device: cuda

--- Fold 1/5 -----------------------------
 subtype_counts = [32.0, 22.0, 67.0, 25.0, 249.0]
 effective_num = [31.95044956406009, 21.976915392687623, 66.77937828448458, 24.9700229873556, 245.93766566233842]
 weights_ens (raw) = [7.697471209888228, 11.190727236642555, 3.682838504644767, 9.84931675020392, 1.0]
 subtype_class_weights (normalized) = [7.6975, 11.1907, 3.6828, 9.8493, 1.0]
[FOLD 1] benign(0)=2077 malignant(1)=395 pos_weight=5.2582
  subtype_class_weights = [7.697500228881836, 11.190699577331543, 3.682800054550171, 9.849300384521484, 1.0]
 [Loss 학습]loss_m=0.4737, loss_s=3.1778, loss_total=1.2060
  Epoch 01/30  loss=1.2060|0.4217  auc=0.9514|0.9838  f1=0.6834|0.7510  oracle_acc=0.2608|0.1939 oracle_f1=0.1835|0.1340 cascade_acc=0.2380|0.1939 loss_slice=3.2247|0.6412 patience=0/7
 [Loss 학습]loss_m=0.2500, loss_s=2.7235, loss_total=0.8782
  Epoch 02/30  loss=0.8782|0.3818  auc=0.9860|0.9905  f1=0.8258|0.8585  oracle_acc=0.3848|0.4184 orac

KeyboardInterrupt: 

In [ ]:
#13 5-fold 평균 +- 표준편차

all_cv_df = pd.concat([resnet_cv_df, vit_cv_df, hybrid_cv_df], ignore_index=True)

summary_df = (
    all_cv_df.groupby("model")[["mal auc", "mal f1", "mal recall", "subtype_oracle_acc", "subtype_oracle_macro_f1", "subtype_cascade_acc", "subtype_gate_recall"]]
    .agg(["mean", "std"])
)

display(summary_df)

In [ ]:
import matplotlib.pyplot as plt

def plot_one_model_by_fold(history_df, model_name):
    folds = sorted(history_df["fold"].unique())

    plt.figure(figsize=(12, 5))
    
    plt.subplot(1,2,1)
    for fold in folds:
        df_fold = history_df[history_df["fold"] == fold].sort_values("epoch")
        plt.plot(df_fold["epoch"], df_fold['train_loss'], linestyle='--', alpha = 0.7, label=f'Fold{fold} Train')
        plt.plot(df_fold["epoch"], df_fold['val_loss'], alpha = 0.7, label=f'Fold{fold} Val')
    plt.title(f"{model_name} - Loss by Fold")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend(fontsize=8)

    plt.subplot(1, 2, 2)
    for fold in folds:
        df_fold = history_df[history_df["fold"] == fold].sort_values("epoch")
        plt.plot(df_fold["epoch"], df_fold["val_auc"], alpha = 0.8, label = f"Fold{fold}")

    plt.title(f"{model_name} - Val ROC-AUC by Fold")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_one_model_by_fold(resnet_history_df, "ResNet50")
plot_one_model_by_fold(vit_history_df, "ViT")
plot_one_model_by_fold(hybrid_history_df, "Hybrid")